# Lesson 9: 方言口音识别教程

本教程演示如何使用 wav2vec 2.0 进行方言口音分类。

## 学习目标
- 理解口音识别任务
- 学习使用 wav2vec 2.0 进行分类
- 掌握迁移学习在语音任务中的应用
- 实现端到端的口音分类系统

## 1. 环境准备

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torchaudio
from transformers import Wav2Vec2Model, Wav2Vec2Processor

from src.models.accent_classifier import AccentClassifier

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print("环境准备完成！")

## 2. 口音识别原理

### 任务定义
给定一段语音，识别说话人的口音类别（如粤语口音、闽南语口音、普通话等）。

### 模型架构
```
音频输入 (16kHz)
    ↓
wav2vec 2.0 特征提取器（冻结）
    ↓
全局平均池化
    ↓
分类头（全连接层）
    ↓
口音类别
```

### 关键技术
- **迁移学习**: 使用预训练的 wav2vec 2.0
- **特征提取器冻结**: 只训练分类头
- **全局池化**: 将变长序列转换为固定维度

## 3. 数据准备

数据格式：
```csv
audio_path,accent
audio1.wav,cantonese
audio2.wav,mandarin
audio3.wav,hakka
```

In [ ]:
# 示例数据
sample_data = pd.DataFrame({
    'audio_path': ['audio1.wav', 'audio2.wav', 'audio3.wav', 'audio4.wav'],
    'accent': ['cantonese', 'mandarin', 'hakka', 'cantonese']
})

print("示例数据:")
display(sample_data)

# 统计口音分布
print("\n口音分布:")
print(sample_data['accent'].value_counts())

# 创建标签映射
accent_labels = sorted(sample_data['accent'].unique())
label2id = {label: i for i, label in enumerate(accent_labels)}
id2label = {i: label for label, i in label2id.items()}

print(f"\n标签映射: {label2id}")

## 4. 加载预训练模型

In [ ]:
model_name = "facebook/wav2vec2-base"

# 加载 processor
processor = Wav2Vec2Processor.from_pretrained(model_name)

print(f"✓ Processor 加载完成")
print(f"采样率: {processor.feature_extractor.sampling_rate} Hz")

## 5. 构建分类模型

In [ ]:
# 模型配置
config = {
    'model_name': model_name,
    'num_labels': len(accent_labels),
    'hidden_size': 768,
    'dropout': 0.1
}

# 创建模型
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AccentClassifier(config)
model.build()
model = model.to(device)

print("模型架构:")
print(model.model)

# 统计参数
total_params = sum(p.numel() for p in model.model.parameters())
trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)

print(f"\n总参数: {total_params:,}")
print(f"可训练参数: {trainable_params:,}")
print(f"可训练比例: {trainable_params/total_params:.2%}")

## 6. 数据预处理

In [ ]:
def load_and_preprocess_audio(audio_path, processor, target_sr=16000):
    """加载和预处理音频"""
    # 加载音频
    speech_array, sampling_rate = torchaudio.load(audio_path)
    
    # 重采样
    if sampling_rate != target_sr:
        resampler = torchaudio.transforms.Resample(sampling_rate, target_sr)
        speech_array = resampler(speech_array)
    
    # 转换为单声道
    if speech_array.shape[0] > 1:
        speech_array = torch.mean(speech_array, dim=0, keepdim=True)
    
    speech_array = speech_array.squeeze().numpy()
    
    # 使用 processor 处理
    inputs = processor(
        speech_array,
        sampling_rate=target_sr,
        return_tensors='pt',
        padding=True
    )
    
    return inputs

print("数据预处理函数定义完成！")

## 7. 推理演示

In [ ]:
def predict_accent(audio_path, model, processor, id2label, device):
    """预测口音"""
    # 预处理
    inputs = load_and_preprocess_audio(audio_path, processor)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # 推理
    model.eval()
    with torch.no_grad():
        logits = model.predict(inputs['input_values'])
        probs = torch.nn.functional.softmax(logits, dim=-1)
        pred_id = torch.argmax(probs, dim=-1).item()
    
    # 获取预测结果
    pred_label = id2label[pred_id]
    confidence = probs[0, pred_id].item()
    
    return pred_label, confidence, probs[0].cpu().numpy()

# 示例推理（如果有音频文件）
print("推理函数定义完成！")
print("\n如果有训练好的模型，可以这样使用：")
print("pred_label, confidence, probs = predict_accent('test.wav', model, processor, id2label, device)")
print("print(f'预测口音: {pred_label}, 置信度: {confidence:.2%}')")

## 8. 训练流程（概念演示）

In [ ]:
training_pseudocode = """
# 训练流程
from torch.utils.data import DataLoader
from src.training.accent_trainer import AccentTrainer

# 1. 准备数据
train_dataset = AccentDataset(train_df, processor, label2id)
val_dataset = AccentDataset(val_df, processor, label2id)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

# 2. 配置训练器
config = {
    'epochs': 20,
    'learning_rate': 1e-4,
    'patience': 5,
    'device': device
}

trainer = AccentTrainer(model, config)

# 3. 训练
history = trainer.train(train_loader, val_loader)

# 4. 保存模型
model.save('checkpoints/accent_classifier/best_model.pth')
"""

print("训练流程（伪代码）:")
print(training_pseudocode)

print("\n实际训练命令:")
print("python scripts/lesson_09_accent_recognition.py --mode train \\")
print("    --data_dir material/lesson_9/audio \\")
print("    --label_file material/lesson_9/labels.csv \\")
print("    --model_name facebook/wav2vec2-base \\")
print("    --output_dir checkpoints/accent_classifier \\")
print("    --epochs 20")

## 9. 可视化预测结果

In [ ]:
# 模拟预测概率
sample_probs = np.array([0.7, 0.2, 0.1])  # cantonese, hakka, mandarin
labels = ['cantonese', 'hakka', 'mandarin']

# 绘制概率分布
plt.figure(figsize=(10, 6))
bars = plt.bar(labels, sample_probs, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
plt.xlabel('口音类别', fontsize=12)
plt.ylabel('概率', fontsize=12)
plt.title('口音识别预测概率分布', fontsize=14)
plt.ylim(0, 1)

# 添加数值标签
for bar, prob in zip(bars, sample_probs):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{prob:.2%}',
             ha='center', va='bottom', fontsize=11)

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"预测结果: {labels[np.argmax(sample_probs)]}")
print(f"置信度: {np.max(sample_probs):.2%}")

## 10. 混淆矩阵分析

In [ ]:
# 模拟混淆矩阵
from sklearn.metrics import confusion_matrix

# 模拟数据
y_true = [0, 0, 1, 1, 2, 2, 0, 1, 2, 0]
y_pred = [0, 0, 1, 2, 2, 2, 0, 1, 1, 1]

cm = confusion_matrix(y_true, y_pred)

# 绘制混淆矩阵
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels,
            yticklabels=labels)
plt.xlabel('预测标签', fontsize=12)
plt.ylabel('真实标签', fontsize=12)
plt.title('口音识别混淆矩阵', fontsize=14)
plt.tight_layout()
plt.show()

# 计算准确率
accuracy = np.trace(cm) / np.sum(cm)
print(f"\n准确率: {accuracy:.2%}")

## 11. 特征可视化（t-SNE）

In [ ]:
from sklearn.manifold import TSNE

# 模拟特征（实际应该从模型中提取）
np.random.seed(42)
features = np.random.randn(60, 768)  # 60 个样本，768 维特征
labels_array = np.repeat([0, 1, 2], 20)  # 每类 20 个样本

# t-SNE 降维
tsne = TSNE(n_components=2, random_state=42)
features_2d = tsne.fit_transform(features)

# 绘制
plt.figure(figsize=(10, 8))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

for i, label in enumerate(labels):
    mask = labels_array == i
    plt.scatter(features_2d[mask, 0], features_2d[mask, 1],
               c=colors[i], label=label, s=100, alpha=0.6, edgecolors='black')

plt.xlabel('t-SNE 维度 1', fontsize=12)
plt.ylabel('t-SNE 维度 2', fontsize=12)
plt.title('口音特征 t-SNE 可视化', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 12. 总结

本教程演示了：
1. ✅ 口音识别任务定义
2. ✅ 使用 wav2vec 2.0 进行分类
3. ✅ 迁移学习的应用
4. ✅ 模型推理和评估
5. ✅ 结果可视化

### 关键技术点
- **迁移学习**: 利用预训练模型的通用表示
- **特征提取器冻结**: 只训练分类头，加快训练
- **全局池化**: 处理变长序列
- **低资源学习**: 适合小数据集

### 实际训练
```bash
python scripts/lesson_09_accent_recognition.py --mode train \
    --data_dir material/lesson_9/audio \
    --label_file material/lesson_9/labels.csv \
    --model_name facebook/wav2vec2-base \
    --output_dir checkpoints/accent_classifier \
    --epochs 20 \
    --batch_size 8
```

### 下一步
- 收集更多口音数据
- 尝试更大的预训练模型（wav2vec2-large）
- 实现多任务学习（口音+语音识别）
- 添加数据增强（速度扰动、噪声添加）